In [2]:
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
import pandas as pd
import numpy as np


data = pd.read_csv("vehicle.csv")

data.head()

,year,price,transmission,mileage,fuelType,tax,mpg,engineSize,Brand,Car_Type,High_Performance
0,2017,12000,Automatic,15944,Petrol,150.0,57.7,1.0,Ford,Hatchback,0
1,2018,14000,Manual,9083,Petrol,150.0,57.7,1.0,Ford,Hatchback,0
2,2017,13000,Manual,12456,Petrol,150.0,57.7,1.0,Ford,Hatchback,0
3,2019,17500,Manual,10460,Petrol,145.0,40.3,1.5,Ford,Hatchback,0
4,2019,16500,Automatic,1482,Petrol,145.0,48.7,1.0,Ford,Hatchback,0


In [3]:

numeric_features = ['year', 'engineSize', 'mileage', 'tax', 'mpg', 'High_Performance']
categorical_features = ['Brand', 'fuelType', 'Car_Type', 'transmission']


numeric_imputer = SimpleImputer(strategy='median')
data[numeric_features] = numeric_imputer.fit_transform(data[numeric_features])

categorical_imputer = SimpleImputer(strategy='most_frequent')
data[categorical_features] = categorical_imputer.fit_transform(data[categorical_features])


scaler = StandardScaler()
data[numeric_features] = scaler.fit_transform(data[numeric_features])

encoder = OneHotEncoder()
encoded_categories = encoder.fit_transform(data[categorical_features]).toarray()
vehicle_features = np.hstack([data[numeric_features].values, encoded_categories])


In [4]:
!pip install dask_ml

  Using cached dask_ml-2024.4.4-py3-none-any.whl.metadata (5.9 kB)
  Using cached dask_glm-0.3.2-py2.py3-none-any.whl.metadata (1.5 kB)
  Using cached dask-2024.10.0-py3-none-any.whl.metadata (3.7 kB)
  Using cached distributed-2024.10.0-py3-none-any.whl.metadata (3.3 kB)
  Using cached multipledispatch-1.0.0-py3-none-any.whl.metadata (3.8 kB)
  Using cached cloudpickle-3.1.0-py3-none-any.whl.metadata (7.0 kB)
  Using cached fsspec-2024.10.0-py3-none-any.whl.metadata (11 kB)
  Using cached partd-1.4.2-py3-none-any.whl.metadata (4.6 kB)
  Using cached toolz-1.0.0-py3-none-any.whl.metadata (5.1 kB)
  Using cached sparse-0.15.4-py2.py3-none-any.whl.metadata (4.5 kB)
  Using cached dask_expr-1.1.16-py3-none-any.whl.metadata (2.5 kB)
  Using cached locket-1.0.0-py2.py3-none-any.whl.metadata (2.8 kB)
  Using cached sortedcontainers-2.4.0-py2.py3-none-any.whl.metadata (10 kB)
  Using cached tblib-3.0.0-py3-none-any.whl.metadata (25 kB)
  Using cached zict-3.0.0-py2.py3-none-any.whl.metadata (

In [5]:
import dask.array as da
from dask_ml.metrics import pairwise_distances

data_dask = da.from_array(vehicle_features, chunks=(1000, vehicle_features.shape[1]))
data_array = vehicle_features.copy()

In [6]:
similarity_matrix = pairwise_distances(data_dask, data_array, metric= 'cosine')
similarity_matrix

dask.array<pairwise_distances, shape=(107343, 107343), dtype=float64, chunksize=(1000, 107343), chunktype=numpy.ndarray>

In [7]:
similarity_matrix = similarity_matrix.astype(np.float32)
similarity_matrix

dask.array<astype, shape=(107343, 107343), dtype=float32, chunksize=(1000, 107343), chunktype=numpy.ndarray>

In [9]:
from datetime import datetime
from elasticsearch import Elasticsearch

# Connect to 'http://localhost:9200'
client = Elasticsearch("http://localhost:9200")

In [11]:
from elasticsearch import Elasticsearch, helpers


similarity_matrix

# Connect to Elasticsearch
es = Elasticsearch("http://localhost:9200")

# Define the index and mappings
index_name = "similarity_matrix"
if not es.indices.exists(index=index_name):
    es.indices.create(index=index_name, body={
        "mappings": {
            "properties": {
                "item_id": {"type": "integer"},
                "similarities": {"type": "dense_vector", "dims": 1000}
            }
        }
    })

# Prepare bulk upload data
actions = [
    {
        "_index": index_name,
        "_id": i,
        "_source": {
            "item_id": i,
            "similarities": row.tolist()
        }
    }
    for i, row in enumerate(similarity_matrix)
]

# Bulk insert
helpers.bulk(es, actions)

# Query example: Retrieve similarities for a specific item
item_id = 0
result = es.get(index=index_name, id=item_id)
similarities = result["_source"]["similarities"]


ConnectionError: ConnectionError(('Connection aborted.', RemoteDisconnected('Remote end closed connection without response'))) caused by: ProtocolError(('Connection aborted.', RemoteDisconnected('Remote end closed connection without response')))

In [ ]:
# REDIS
import redis
import numpy as np

# Example similarity matrix
similarity_matrix = np.random.rand(1000, 1000).astype(np.float32)

# Connect to Redis
r = redis.StrictRedis(host='localhost', port=6379, db=0)

# Store each row as a hash
for i, row in enumerate(similarity_matrix):
    # Use a unique key for each row, e.g., "similarity:item_id"
    r.hset(f"similarity:{i}", mapping={str(j): float(row[j]) for j in range(len(row))})

# Query example: Retrieve similarities for a specific item
item_id = 0
similarities = r.hgetall(f"similarity:{item_id}")
similarities = {int(k): float(v) for k, v in similarities.items()}


In [14]:
# SQL SERVER
import pyodbc
import numpy as np

# Assuming `similarity_matrix` is a 2D NumPy array
# similarity_matrix = np.load("similarity_matrix.npy")  # Load your precomputed matrix
num_items = similarity_matrix.shape[0]

# Define your SQL Server connection parameters
server = 'DESKTOP-EAGDO5Q\DATAVIZ'  # e.g., 'localhost' or 'your_server_name'
database = 'recommendation'

# Create a connection string for Windows Authentication
conn_string = f'DRIVER={{ODBC Driver 17 for SQL Server}};SERVER={server};DATABASE={database};Trusted_Connection=yes;'

# Connect to SQL Server
conn = pyodbc.connect(conn_string)
cursor = conn.cursor()

# Create a table to store the similarity matrix
cursor.execute('''
    IF OBJECT_ID('similarities', 'U') IS NOT NULL 
        DROP TABLE similarities;
    CREATE TABLE similarities (
        item_id INT,
        similar_item_id INT,
        similarity_score FLOAT
    )
''')

# Insert data row by row
for i in range(num_items):
    for j in range(num_items):
        cursor.execute('INSERT INTO similarities (item_id, similar_item_id, similarity_score) VALUES (?, ?, ?)',
                       (i, j, float(similarity_matrix[i, j])))

# Commit changes and close the connection
conn.commit()
cursor.close()
conn.close()